# 00 — Environment and LADI-v2 dataset cache

This notebook prepares a Kaggle-friendly local cache from the Hugging Face LADI-v2 dataset. Run it first. It writes `train_data.csv`, `val_data.csv`, `test_data.csv`, and image files into `/kaggle/working/ladi_v2a_cache` by default.

The default target is the practical `v2a_resized` label set: 12 labels, resized imagery, and the official train/validation/test split.

In [ ]:

from pathlib import Path
import sys, os, json, time

# Locate the project root whether this folder is in /kaggle/working or uploaded as a Kaggle dataset.
candidates = [Path.cwd(), Path('/kaggle/working/ladi_v2_gcn_project')]
if Path('/kaggle/input').exists():
    candidates += list(Path('/kaggle/input').glob('*'))
PROJECT_ROOT = None
for c in candidates:
    if (c / 'src' / 'ladi_config.py').exists():
        PROJECT_ROOT = c
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find project src/. Upload/unzip the full ladi_v2_gcn_project folder first.')
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
# Kaggle normally already has most packages. Run this cell only if imports fail.
# !pip -q install -U datasets huggingface_hub scikit-learn tqdm pandas pillow matplotlib


In [ ]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image

from src.ladi_config import DATASET_ID, HF_CONFIG, HF_REVISION, LABEL_COLS_V2A, DEFAULT_CACHE_DIR, DEFAULT_HF_BASE_DIR

CACHE_DIR = Path('/kaggle/working/ladi_v2a_cache')
HF_BASE_DIR = Path('/kaggle/working/ladi_hf')
IMG_DIR = CACHE_DIR / 'images'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)

# Set a small number like 200 for a smoke test. Use None for the full project run.
LIMIT_PER_SPLIT = None
print('CACHE_DIR:', CACHE_DIR)


In [ ]:
from datasets import load_dataset

# Preferred route: script revision + v2a_resized because it avoids unnecessary full-resolution downloads.
try:
    ds = load_dataset(
        DATASET_ID,
        HF_CONFIG,
        revision=HF_REVISION,
        streaming=True,
        download_ladi=True,
        base_dir=str(HF_BASE_DIR),
        trust_remote_code=True,
    )
    print('Loaded streaming script dataset:', ds)
except Exception as e:
    print('Script/streaming load failed:', repr(e))
    print('Falling back to the default Hugging Face dataset viewer/parquet route.')
    ds = load_dataset(DATASET_ID)
    print(ds)


In [ ]:
def save_split(split_name: str, out_csv: Path, limit=None):
    rows = []
    iterable = ds[split_name]
    for i, ex in enumerate(tqdm(iterable, desc=f'Caching {split_name}')):
        if limit is not None and i >= limit:
            break
        img = ex['image']
        # Some HF image objects carry a filename; we still create stable local IDs.
        image_id = f'{split_name}_{i:06d}'
        image_path = IMG_DIR / f'{image_id}.jpg'
        if not image_path.exists():
            if not isinstance(img, Image.Image):
                img = Image.open(img).convert('RGB')
            else:
                img = img.convert('RGB')
            img.save(image_path, quality=92)
        row = {'ID': image_id}
        for lab in LABEL_COLS_V2A:
            row[lab] = int(bool(ex[lab]))
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(out_csv.name, df.shape)
    return df

train_df = save_split('train', CACHE_DIR / 'train_data.csv', LIMIT_PER_SPLIT)
val_df = save_split('validation', CACHE_DIR / 'val_data.csv', LIMIT_PER_SPLIT)
test_df = save_split('test', CACHE_DIR / 'test_data.csv', LIMIT_PER_SPLIT)

print('Images saved:', len(list(IMG_DIR.glob('*.jpg'))))
train_df.head()


In [ ]:
# Basic sanity checks
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print('\n', name, df.shape)
    print(df[LABEL_COLS_V2A].mean().sort_values(ascending=False))


Next: run **01_eda_and_label_graphs.ipynb** to inspect label imbalance and build static co-occurrence graphs.